# 3c — Compute Agreement

Loads human coding files, computes human-human reliability, computes human-AI agreement, and writes feature inclusion decisions.

## Step 0 — Imports and paths

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

try:
    from pingouin import intraclass_corr
except ImportError:
    intraclass_corr = None

NOTEBOOK_DIR = Path().resolve()
ANALYSIS_V2 = NOTEBOOK_DIR.parent
PROJECT_ROOT = ANALYSIS_V2.parent
DATA_DIR = ANALYSIS_V2 / 'data'
HUMAN_CODING_DIR = DATA_DIR / 'human_coding'
VALIDATION_DIR = DATA_DIR / 'validation'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'

VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

print('HUMAN_CODING_DIR:', HUMAN_CODING_DIR)
print('VALIDATION_DIR  :', VALIDATION_DIR)


## Step 1 — Helpers and field definitions

In [ ]:
INSTRUMENT_A_FIELDS = [
    'idea_trajectory',
    'idea_trajectory_justification',
    'problem_specificity_level',
    'problem_specificity_justification',
    'decision_crystallization_level',
    'decision_crystallization_justification',
    'ambition_level',
    'cross_disciplinary_bridging',
    'cross_disciplinary_bridging_description',
    'explicit_commitment_signal',
    'commitment_signal_quote',
]
INSTRUMENT_B_FIELDS = [
    'pronoun_shift_flag',
    'shared_vision_indicator',
    'shared_vision_quote',
    'laughter_quality',
    'personal_disclosure',
    'dissent_response_quality',
    'risk_acknowledgment_with_enthusiasm',
    'risk_enthusiasm_quote',
    'meeting_structure_quality',
]
INSTRUMENT_C_FIELDS = [
    'collective_engagement_level',
    'collective_engagement_justification',
]
ALL_FIELDS = INSTRUMENT_A_FIELDS + INSTRUMENT_B_FIELDS + INSTRUMENT_C_FIELDS

ORDINAL_FIELDS = [
    'collective_engagement_level',
    'problem_specificity_level',
    'decision_crystallization_level',
    'dissent_response_quality',
]
BINARY_FIELDS = [
    'explicit_commitment_signal', 'cross_disciplinary_bridging',
    'pronoun_shift_flag', 'shared_vision_indicator',
    'personal_disclosure', 'risk_acknowledgment_with_enthusiasm',
]
CATEGORICAL_FIELDS = [
    'idea_trajectory', 'ambition_level', 'laughter_quality', 'meeting_structure_quality',
]
TIER_1 = [
    'idea_trajectory', 'collective_engagement_level', 'explicit_commitment_signal',
    'decision_crystallization_level', 'pronoun_shift_flag', 'cross_disciplinary_bridging',
    'shared_vision_indicator',
]
TIER_2 = [
    'problem_specificity_level', 'ambition_level', 'laughter_quality',
    'dissent_response_quality', 'risk_acknowledgment_with_enthusiasm',
    'personal_disclosure', 'meeting_structure_quality',
]


def normalize_filename(name: str) -> str:
    name = re.sub(r'[^a-zA-Z0-9._()]', '_', name)
    name = re.sub(r'_+', '_', name)
    return name.strip('_')


def load_table(stem: str) -> pd.DataFrame:
    parquet_path = DATA_DIR / f'{stem}.parquet'
    csv_path = DATA_DIR / f'{stem}.csv'
    if parquet_path.exists():
        try:
            return pd.read_parquet(parquet_path)
        except Exception as e:
            print(f'Falling back to CSV for {stem}: {type(e).__name__}: {e}')
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'Could not find {parquet_path.name} or {csv_path.name}')


def load_rater_codes(rater_dir: Path, instrument: str) -> pd.DataFrame:
    files = sorted((rater_dir / f'instrument_{instrument}').glob('*.csv'))
    dfs = []
    for f in files:
        chunk_id = f.name.replace(f'__instrument_{instrument}.csv', '')
        df = pd.read_csv(f)
        if {'field', 'rater_code'}.issubset(df.columns):
            pivoted = df[['field', 'rater_code']].copy()
            pivoted['chunk_id'] = chunk_id
            dfs.append(pivoted)
    if not dfs:
        return pd.DataFrame()
    stacked = pd.concat(dfs, ignore_index=True)
    return stacked.pivot(index='chunk_id', columns='field', values='rater_code')


def resolve_json_path(row: pd.Series) -> Path:
    conf = row['conference_id']
    session_key = row['session_key']
    chunk_fn = row['chunk_file_name']
    stem_norm = normalize_filename(Path(chunk_fn).stem)
    session_grp = session_key.split('/')[0].replace('-', '_')
    out_dir = OUTPUT_DIR / conf / f'output_{session_grp}'
    if '/' in session_key:
        recording_folder = session_key.split('/', 1)[1]
        base = out_dir / recording_folder
    else:
        base = out_dir
    plain = base / f'{stem_norm}.json'
    attn = base / f'ATTN_{stem_norm}.json'
    return plain if plain.exists() else attn


def compute_icc(scores_long: pd.DataFrame, target_col: str, rater_col: str, rating_col: str):
    if intraclass_corr is None:
        return np.nan
    icc_result = intraclass_corr(data=scores_long, targets=target_col, raters=rater_col, ratings=rating_col)
    row = icc_result.loc[icc_result['Type'] == 'ICC2']
    return row['ICC'].iloc[0] if not row.empty else np.nan


## Step 2 — Load registry and rater files

In [ ]:
registry = load_table('chunk_registry_v2')
val_registry = registry.loc[registry['human_validation_set'] == True].copy()

rater1_dir = HUMAN_CODING_DIR / 'rater1'
rater2_dir = HUMAN_CODING_DIR / 'rater2'

rater1_A = load_rater_codes(rater1_dir, 'A')
rater2_A = load_rater_codes(rater2_dir, 'A')
rater1_B = load_rater_codes(rater1_dir, 'B')
rater2_B = load_rater_codes(rater2_dir, 'B')
rater1_C = load_rater_codes(rater1_dir, 'C')
rater2_C = load_rater_codes(rater2_dir, 'C')

rater1_all = pd.concat([rater1_A, rater1_B, rater1_C], axis=1).reindex(columns=ALL_FIELDS)
rater2_all = pd.concat([rater2_A, rater2_B, rater2_C], axis=1).reindex(columns=ALL_FIELDS)

print('Validation chunks:', len(val_registry))
print('Rater1 coded chunks:', len(rater1_all))
print('Rater2 coded chunks:', len(rater2_all))


## Step 3 — Resolve disagreements

In [ ]:
resolved = rater1_all.copy()
disagreements = []

for field in ALL_FIELDS:
    if field not in rater1_all.columns or field not in rater2_all.columns:
        continue
    mismatch = (
        rater1_all[field].fillna('__MISSING__') != rater2_all[field].fillna('__MISSING__')
    )
    disagreements.append({
        'field': field,
        'n_disagreements': int(mismatch.sum()),
        'pct_disagreements': float(mismatch.mean()) if len(mismatch) else np.nan,
    })
    resolved.loc[mismatch, field] = 'DISPUTED'

disagreement_df = pd.DataFrame(disagreements)
disagreement_path = HUMAN_CODING_DIR / 'disagreement_summary.csv'
resolved_path = HUMAN_CODING_DIR / 'resolved_codes.csv'
disagreement_df.to_csv(disagreement_path, index=False)
resolved.to_csv(resolved_path)

print('Saved:', disagreement_path)
print('Saved:', resolved_path)
disagreement_df.sort_values('pct_disagreements', ascending=False).head(10)


## Step 4 — Human-human reliability

In [ ]:
irr_results = []

for field in BINARY_FIELDS + CATEGORICAL_FIELDS:
    paired = rater1_all[[field]].join(rater2_all[[field]], lsuffix='_r1', rsuffix='_r2').dropna()
    if paired.empty:
        continue
    kappa = cohen_kappa_score(paired[f'{field}_r1'], paired[f'{field}_r2'])
    irr_results.append({'field': field, 'metric': 'cohen_kappa', 'value': kappa, 'n': len(paired)})

for field in ORDINAL_FIELDS:
    scores_long = pd.concat([
        rater1_all[[field]].rename(columns={field: 'rating'}).assign(rater='rater1').reset_index(),
        rater2_all[[field]].rename(columns={field: 'rating'}).assign(rater='rater2').reset_index(),
    ])
    scores_long['rating'] = pd.to_numeric(scores_long['rating'], errors='coerce')
    scores_long = scores_long.dropna(subset=['rating'])
    if scores_long.empty:
        continue
    icc_val = compute_icc(scores_long, target_col='chunk_id', rater_col='rater', rating_col='rating')
    irr_results.append({'field': field, 'metric': 'ICC2', 'value': icc_val, 'n': scores_long['chunk_id'].nunique()})

irr_df = pd.DataFrame(irr_results)
irr_path = VALIDATION_DIR / 'human_irr_results.csv'
irr_df.to_csv(irr_path, index=False)
print('Saved:', irr_path)
irr_df.sort_values('value')


## Step 5 — Human-AI agreement

In [ ]:
resolved_human = pd.read_csv(resolved_path, index_col='chunk_id') if resolved_path.exists() else pd.DataFrame()

ai_rows = []
for _, row in val_registry.iterrows():
    path = resolve_json_path(row)
    if not path.exists():
        continue
    try:
        with open(path) as f:
            data = json.load(f)
    except Exception:
        continue
    summary = data.get('chunk_summary', {}).copy()
    summary['chunk_id'] = row['chunk_id']
    ai_rows.append(summary)

ai_df = pd.DataFrame(ai_rows).set_index('chunk_id') if ai_rows else pd.DataFrame()

shared = resolved_human.index.intersection(ai_df.index)
human_aligned = resolved_human.loc[shared] if len(shared) else pd.DataFrame()
ai_aligned = ai_df.loc[shared] if len(shared) else pd.DataFrame()

hai_results = []

for field in BINARY_FIELDS + CATEGORICAL_FIELDS:
    if field not in human_aligned.columns or field not in ai_aligned.columns:
        continue
    paired = human_aligned[[field]].join(ai_aligned[[field]], lsuffix='_human', rsuffix='_ai').dropna()
    paired = paired[(paired[f'{field}_human'] != 'DISPUTED') & (paired[f'{field}_ai'] != 'DISPUTED')]
    if paired.empty:
        continue
    kappa = cohen_kappa_score(paired[f'{field}_human'], paired[f'{field}_ai'])
    hai_results.append({'field': field, 'metric': 'cohen_kappa_human_ai', 'value': kappa, 'n': len(paired)})

for field in ORDINAL_FIELDS:
    if field not in human_aligned.columns or field not in ai_aligned.columns:
        continue
    scores_long = pd.concat([
        human_aligned[[field]].rename(columns={field: 'rating'}).assign(source='human').reset_index(),
        ai_aligned[[field]].rename(columns={field: 'rating'}).assign(source='ai').reset_index(),
    ])
    scores_long['rating'] = pd.to_numeric(scores_long['rating'].replace('DISPUTED', np.nan), errors='coerce')
    scores_long = scores_long.dropna(subset=['rating'])
    if scores_long.empty:
        continue
    icc_val = compute_icc(scores_long, target_col='chunk_id', rater_col='source', rating_col='rating')
    hai_results.append({'field': field, 'metric': 'ICC2_human_ai', 'value': icc_val, 'n': scores_long['chunk_id'].nunique()})

hai_df = pd.DataFrame(hai_results)
hai_path = VALIDATION_DIR / 'human_ai_agreement.csv'
hai_df.to_csv(hai_path, index=False)
print('Saved:', hai_path)
hai_df.sort_values('value')


## Step 6 — Feature inclusion decision rule

In [ ]:
KAPPA_INCLUDE = 0.60
KAPPA_CAVEAT = 0.40

decisions = []
for _, row in hai_df.iterrows():
    field = row['field']
    k = row['value']
    tier = 'tier1' if field in TIER_1 else ('tier2' if field in TIER_2 else 'tier3')
    if pd.isna(k):
        decision = 'insufficient_data'
    elif k >= KAPPA_INCLUDE:
        decision = 'include'
    elif k >= KAPPA_CAVEAT:
        decision = 'include_with_caveat'
    else:
        decision = 'exclude'
    decisions.append({'field': field, 'tier': tier, 'human_ai_kappa': k, 'decision': decision})

feature_decisions = pd.DataFrame(decisions)
decision_path = VALIDATION_DIR / 'feature_inclusion_decision.csv'
feature_decisions.to_csv(decision_path, index=False)

print('Saved:', decision_path)
display(feature_decisions.groupby(['tier', 'decision']).size().rename('n'))
feature_decisions.sort_values(['tier', 'human_ai_kappa'])
